In [3]:
import pandas as pd
import os
from tqdm import tqdm

# Input paths
SOURCE_CSV = "/home/yuyao/methane_train/csvs/merged_file.csv" 
OUT_CSV = "./CM_L89_L2SR_gee.csv"

# Directory mapping
DIRS = {
    "t0": "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_l89_L2SR/raw_data_dir_L89_L2SR_by_gee",
    "m7": "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_l89_L2SR/raw_data_dir_L89_-7_L2SR_by_gee",
    "hist": "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_l89_L2SR/raw_data_dir_L89_90360_L2SR_by_gee"
}

def build_file_map(directory):
    """Creates a mapping of plume_id -> [list of full paths]"""
    file_map = {}
    if not os.path.exists(directory):
        print(f"Warning: Directory not found: {directory}")
        return file_map
    
    # os.scandir is the fastest way to list files
    with os.scandir(directory) as it:
        for entry in it:
            if entry.is_file() and entry.name.endswith(".tif"):
                # Extract plume_id: assumes plume_id is the string before the first underscore 
                # or based on your example 'tan...-A'
                if "_minus90" in entry.name:
                    pid = entry.name.split('_l89')[0].split('_minus90')[0]
                elif "_minus360" in entry.name:
                    pid = entry.name.split('_l89')[0].split('_minus360')[0]
                else:
                    pid = entry.name.split('_l89')[0]
                # pid = entry.name.split('_l89')[0].split('_minus90')[0] or entry.name.split('_l89')[0].split('_minus360')[0]
                if pid not in file_map:
                    file_map[pid] = []
                file_map[pid].append(entry.path)
    return file_map

# 1. Pre-scan all directories (This is the speed secret)
print("Scanning directories...")
map_t0 = build_file_map(DIRS["t0"])
map_m7 = build_file_map(DIRS["m7"])
map_hist = build_file_map(DIRS["hist"])

# 2. Load source and match in memory
df = pd.read_csv(SOURCE_CSV)
new_data = []

print("Matching plumes...")
for pid in tqdm(df['plume_id'].unique()):
    # Lookup t0
    path0 = map_t0.get(pid, [""])[0]
    
    # Lookup m7
    path7 = map_m7.get(pid, [""])[0]
    
    # Lookup hist (90 and 360)
    hist_files = map_hist.get(pid, [])
    path90 = ""
    path360 = ""
    
    for f in hist_files:
        print(f)
        # exit()
        if "minus90_" in f:
            path90 = f
        elif "minus360_" in f:
            # Assumes the other file in the 90360 folder is the 360 one
            path360 = f

    new_data.append({
        "plume_id": pid,
        "l89_path": path0,
        "l89_-7_path": path7,
        "l89_pre_path": path90,
        "l89_pre_pre_path": path360,
        "std_ok": 0
    })

# 3. Save
out_df = pd.DataFrame(new_data)
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
out_df.to_csv(OUT_CSV, index=False)

print(f"Done. Successfully matched {len(out_df)} plumes.")

Scanning directories...
Matching plumes...


100%|██████████| 24470/24470 [00:00<00:00, 546798.47it/s]

/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_l89_L2SR/raw_data_dir_L89_90360_L2SR_by_gee/GAO20191022t151126p0000-D_minus360_l89_sr_LANDSAT_8_20181028T172713Z.tif
/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_l89_L2SR/raw_data_dir_L89_90360_L2SR_by_gee/GAO20191022t151126p0000-D_minus90_l89_sr_LANDSAT_8_20190727T172720Z.tif
/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_l89_L2SR/raw_data_dir_L89_90360_L2SR_by_gee/GAO20191022t151126p0000-F_minus90_l89_sr_LANDSAT_8_20190727T172720Z.tif
/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_l89_L2SR/raw_data_dir_L89_90360_L2SR_by_gee/GAO20191022t151126p0000-F_minus360_l89_sr_LANDSAT_8_20181028T172713Z.tif
/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_l89_L2SR/raw_data_dir_L89_90360_L2SR_by_gee/GAO20191022t151126p0000-G_minus90_l89_sr_LANDSAT_8_20190727T172720Z.tif
/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/raw_data_dir_l89_L2SR/raw_data_dir_

In [11]:
df = pd.read_csv("./CM_L89_L2SR_gee.csv")
df.dropna(subset=['l89_path', 'l89_-7_path', 'l89_pre_path', 'l89_pre_pre_path'], inplace=True)
print(len(df))
df.to_csv("./CM_L89_L2SR_gee.csv", index=False)

2623


In [3]:
import os
import time
import threading
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import contextlib
from collections import Counter, deque

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from rasterio.transform import from_origin
import tifffile
from tqdm import tqdm

# =========================
# Config (L89 Specific)
# =========================
CM_CSV  = "./CM_L89_L2SR_gee.csv"
OUT_CSV = "./CM_L89_L2SR_std512.csv"

# Target directory for the 512x512 crops
OUT_ROOT = "/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/data_dir_l89_L2SR/L89_-790360_512std"

WINDOW_SIZE = 512
MAX_WORKERS = 12
FLUSH_EVERY_SEC = 60

# Requirements: t0 and m7/m8 are usually the priority
REQUIRE_T0 = True
REQUIRE_M7 = True 

# =========================
# Utils
# =========================
def debug(msg: str) -> None:
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"[{ts}][pid:{os.getpid()}][tid:{threading.get_ident()}] {msg}", flush=True)

def safe_mkdir(p: str) -> None:
    os.makedirs(p, exist_ok=True)

@contextlib.contextmanager
def suppress_stderr():
    with open(os.devnull, "w") as devnull:
        with contextlib.redirect_stderr(devnull):
            yield

# =========================
# Array helpers -> BHW with NaN handling
# =========================
def to_bhw_l89(arr: np.ndarray) -> np.ndarray:
    """
    Standardizes to (B, H, W). 
    Landsat 8/9 SR from GEE typically comes as 7 bands.
    """
    if arr.ndim == 2:
        return arr[None, :, :]
    
    # If HWC (7 bands at end), transpose to CHW
    if arr.shape[-1] == 7 and arr.shape[0] != 7:
        return np.transpose(arr, (2, 0, 1))
    return arr

def center_crop_or_pad_bhw(bhw: np.ndarray, out_size=512):
    B, H, W = bhw.shape
    
    # If exact size
    if H == out_size and W == out_size:
        return bhw, (0, 0), "noop"

    # Create output array filled with NaNs (since we are using float32)
    out = np.full((B, out_size, out_size), np.nan, dtype=np.float32)
    
    if H < out_size or W < out_size:
        y0 = max(0, (out_size - H) // 2)
        x0 = max(0, (out_size - W) // 2)
        # Only copy what we have
        h_copy = min(H, out_size)
        w_copy = min(W, out_size)
        out[:, y0:y0+h_copy, x0:x0+w_copy] = bhw[:, :h_copy, :w_copy]
        return out, (x0, y0), f"pad_center {H}x{W}"

    # Larger -> center crop
    y0 = (H - out_size) // 2
    x0 = (W - out_size) // 2
    out = bhw[:, y0:y0+out_size, x0:x0+out_size]
    return out, (x0, y0), f"crop_center {H}x{W}"

# =========================
# Core Processing Logic
# =========================
def std_to_512_l89(in_path: str, out_path: str, tag: str, bug_list: list):
    if not (isinstance(in_path, str) and len(in_path) > 0 and os.path.exists(in_path)):
        return False, f"{tag}_missing"

    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        return True, "skipped_exists"

    try:
        with rasterio.open(in_path) as ds:
            # Try to read with window to save memory
            h, w = ds.height, ds.width
            if h >= WINDOW_SIZE and w >= WINDOW_SIZE:
                col0 = (w - WINDOW_SIZE) // 2
                row0 = (h - WINDOW_SIZE) // 2
                win = Window(col0, row0, WINDOW_SIZE, WINDOW_SIZE)
                arr = ds.read(window=win)
            else:
                arr = ds.read()
            
            # Convert to float32 immediately to support NaNs
            bhw = arr.astype(np.float32)
            
            # ✅ IMPORTANT: Handle '0' as NoData for Landsat
            # We assume if all bands are 0 at a pixel, it is a fill value
            nodata_mask = (np.sum(bhw, axis=0) == 0)
            for b in range(bhw.shape[0]):
                bhw[b, nodata_mask] = np.nan

            bhw = to_bhw_l89(bhw)
            bhw2, _, note = center_crop_or_pad_bhw(bhw, out_size=WINDOW_SIZE)

            # Write as multi-band GeoTIFF
            write_gdal_multiband_tif(out_path, bhw2)
            return True, note
            
    except Exception as e:
        bug_list.append(f"{tag}_err: {str(e)}")
        return False, f"{tag}_fail"

def write_gdal_multiband_tif(out_path: str, bhw: np.ndarray):
    B, H, W = bhw.shape
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    
    profile = {
        "driver": "GTiff",
        "height": H, "width": W, "count": B,
        "dtype": "float32",
        "transform": from_origin(0, 0, 1, 1),
        "compress": "deflate",
        "predictor": 2, # Standard for floating point
        "tiled": True,
        "BIGTIFF": "IF_SAFER",
    }
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(bhw)

# =========================
# Worker
# =========================
def process_one_row_l89(row: dict):
    plume_id = str(row.get("plume_id", "")).strip()
    
    # Mapping the paths based on your L89 column names
    p0   = row.get("l89_path", "")
    pm7  = row.get("l89_-7_path", "")
    pm90 = row.get("l89_pre_path", "")
    pm360= row.get("l89_pre_pre_path", "")

    out_dir = os.path.join(OUT_ROOT, plume_id)
    safe_mkdir(out_dir)

    outputs = {
        "l89_0_std_512": os.path.join(out_dir, "l89_0_std_512.tif"),
        "l89_-7_std_512": os.path.join(out_dir, "l89_-7_std_512.tif"),
        "l89_-90_std_512": os.path.join(out_dir, "l89_-90_std_512.tif"),
        "l89_-360_std_512": os.path.join(out_dir, "l89_-360_std_512.tif")
    }

    bug = []
    # Process t0
    ok0, _ = std_to_512_l89(p0, outputs["l89_0_std_512"], "t0", bug)
    if REQUIRE_T0 and not ok0:
        return plume_id, {"std_ok": 0, "std_reason": "t0_fail", "bug": "; ".join(bug)}

    # Process others
    ok7, _ = std_to_512_l89(pm7, outputs["l89_-7_std_512"], "m7", bug)
    if REQUIRE_M7 and not ok7:
        return plume_id, {"std_ok": 0, "std_reason": "m7_fail", "bug": "; ".join(bug)}

    ok90, _ = std_to_512_l89(pm90, outputs["l89_-90_std_512"], "m90", bug)
    ok360, _ = std_to_512_l89(pm360, outputs["l89_-360_std_512"], "m360", bug)

    return plume_id, {
        "std_ok": 1,
        "bug": "; ".join(bug),
        **outputs,
        "has_m90": int(ok90),
        "has_m360": int(ok360)
    }

# =========================
# Main Execution
# =========================
if __name__ == "__main__":
    debug(f"Starting L89 Preprocessing: {CM_CSV}")
    df = pd.read_csv(CM_CSV)
    
    # Initialize columns if not exist
    for c in ["std_ok", "bug", "l89_0_std_512", "l89_-7_std_512", "l89_-90_std_512", "l89_-360_std_512"]:
        if c not in df.columns: df[c] = ""

    work_df = df[df["std_ok"].astype(str) != "1"].copy()
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futs = [ex.submit(process_one_row_l89, r.to_dict()) for _, r in work_df.iterrows()]
        for fut in tqdm(as_completed(futs), total=len(futs), desc="Processing L89"):
            pid, upd = fut.result()
            mask = (df["plume_id"].astype(str) == str(pid))
            for k, v in upd.items():
                df.loc[mask, k] = v
    
    df.to_csv(OUT_CSV, index=False)
    debug("Done.")

[2026-01-26 14:55:55][pid:934926][tid:123214885725248] Starting L89 Preprocessing: ./CM_L89_L2SR_gee.csv


Processing L89:  14%|█▍        | 365/2623 [00:00<00:05, 440.20it/s]/home/yuyao/anaconda3/envs/methane/lib/python3.13/site-packages/rasterio/__init__.py:366: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(
/home/yuyao/anaconda3/envs/methane/lib/python3.13/site-packages/rasterio/__init__.py:366: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore this matrix and save no geotransform without raising an error. This behavior is somewhat driver-specific.
  dataset = writer(
Processing L89:  16%|█▌        | 417/2623 [00:03<00:39, 55.33it/s] /home/yuyao/anaconda3/envs/methane/lib/python3.13/site-packages/rasterio/__init__.py:366: NotGeoreferencedWarning: The given matrix is equal to Affine.identity or its flipped counterpart. GDAL may ignore th

[2026-01-26 15:20:11][pid:934926][tid:123214885725248] Done.


In [ ]:
import pandas as pd
import os
import numpy as np
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.transform import Affine
from pyproj import Transformer
from tqdm import tqdm

# =========================
# Config
# =========================
# The CSV containing plume_id, lat, lon
SOURCE_CSV = '/data2/yuyao/methane_emission/carbon_mapper_data/csvs/merged_file.csv'
# Where the original airborne plume.tif files are stored
BASE_MASK_DIR = '/data2/yuyao/methane_emission/carbon_mapper_data_masks'
# Where to save the 30m, 512x512 aligned masks
OUT_MASK_ROOT = '/mnt/engg-leung/Research_No9_Methane_Emissions/Yuyao/Dataset/data_dir_l89_L2SR/plume_masks_l89_512'

TARGET_RES = 30  # Landsat 8/9 native resolution
TARGET_SIZE = 512

# =========================
# Processing Functions
# =========================

def process_l89_mask(plume_id, center_lat, center_lon):
    plume_dir = os.path.join(BASE_MASK_DIR, plume_id)
    input_tif = os.path.join(plume_dir, 'plume.tif')
    
    # Final output path
    dst_path = os.path.join(OUT_MASK_ROOT, plume_id, 'mask_30m_512.tif')
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)

    if not os.path.exists(input_tif):
        return False, "Input plume.tif missing"

    try:
        with rasterio.open(input_tif) as src:
            if src.height == 0 or src.width == 0:
                return False, "Empty source TIFF"

            # 1. Setup Transformer to convert Lat/Lon to the Mask's local UTM/Projection
            transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
            center_x, center_y = transformer.transform(center_lon, center_lat)

            # 2. Grid Snapping: Align the center to the nearest 30m interval
            # This ensures the mask pixels and satellite pixels overlap exactly.
            center_x = round(center_x / TARGET_RES) * TARGET_RES
            center_y = round(center_y / TARGET_RES) * TARGET_RES

            # 3. Define the 512x512 Window Transform
            # Top-left corner calculation
            top_left_x = center_x - (TARGET_SIZE // 2) * TARGET_RES
            top_left_y = center_y + (TARGET_SIZE // 2) * TARGET_RES
            
            new_transform = Affine(TARGET_RES, 0, top_left_x, 0, -TARGET_RES, top_left_y)
            
            # 4. Prepare Output Array (BHW)
            # Using uint8 as per your original mask logic
            out_mask = np.zeros((1, TARGET_SIZE, TARGET_SIZE), dtype='uint8')

            # 5. Reproject and Resample
            # We reproject directly from the source to the target 512x512 grid.
            # Resampling.nearest is critical for masks to prevent "ghost" values between 0 and 1.
            reproject(
                source=rasterio.band(src, 1),
                destination=out_mask[0],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=new_transform,
                dst_crs=src.crs,
                resampling=Resampling.nearest
            )

            # 6. Write to Disk
            profile = src.profile
            profile.update(
                driver='GTiff',
                transform=new_transform,
                width=TARGET_SIZE,
                height=TARGET_SIZE,
                count=1,
                dtype='uint8',
                compress='lzw'
            )

            with rasterio.open(dst_path, 'w', **profile) as dst:
                dst.write(out_mask)
            
            return True, "Success"

    except Exception as e:
        return False, str(e)

# =========================
# Main Execution Loop
# =========================

if __name__ == "__main__":
    print(f"Loading source data from {SOURCE_CSV}...")
    df = pd.read_csv(SOURCE_CSV)
    
    results = []
    
    # Filtering for valid plume_tif rows
    work_df = df[df['plume_tif'].notna()].copy()
    
    print(f"Total plumes to process: {len(work_df)}")

    for index, row in tqdm(work_df.iterrows(), total=len(work_df)):
        pid = row['plume_id']
        lat = row['plume_latitude']
        lon = row['plume_longitude']
        
        # Skip known bad IDs if necessary (as in your original code)
        if pid == 'GAO20210515t145818p0000-1':
            continue
            
        success, msg = process_l89_mask(pid, lat, lon)
        if not success:
            results.append({"plume_id": pid, "error": msg})

    # Optional: Save a log of failed mask generations
    if results:
        error_df = pd.DataFrame(results)
        error_df.to_csv('mask_generation_errors.csv', index=False)
        print(f"Done. Processed with {len(results)} errors (see mask_generation_errors.csv).")
    else:
        print("Done. All masks generated successfully.")

FileNotFoundError: [Errno 2] No such file or directory: './CM_L89_L2SR_std512.csv'